###initializes a SparkSession

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

### list of files in the DBFS

In [0]:
%fs ls dbfs:///FileStore/tables/

path,name,size,modificationTime
dbfs:/FileStore/tables/Customers__1_-1.csv,Customers__1_-1.csv,1043,1710705699000
dbfs:/FileStore/tables/Customers__1_.csv,Customers__1_.csv,1043,1710705669000
dbfs:/FileStore/tables/Drugs_package.csv,Drugs_package.csv,27234691,1707889380000
dbfs:/FileStore/tables/Drugs_product.csv,Drugs_product.csv,38743242,1707889380000
dbfs:/FileStore/tables/EmployeeSale__1_.csv,EmployeeSale__1_.csv,2736,1710705733000
dbfs:/FileStore/tables/Employee__1_.csv,Employee__1_.csv,385,1710705688000
dbfs:/FileStore/tables/Gender_crosswalk.txt,Gender_crosswalk.txt,72,1709286187000
dbfs:/FileStore/tables/Products__1_.csv,Products__1_.csv,367,1710705650000
dbfs:/FileStore/tables/Relationship_crosswalk.txt,Relationship_crosswalk.txt,264,1709286188000
dbfs:/FileStore/tables/Sales__1_.csv,Sales__1_.csv,2714,1710705719000


###Reading  CSV file into a Spark DataFrame

In [0]:
df_csv2 = spark.read.format("csv").option('header' , True).option('inferSchema', True).load("dbfs:/FileStore/tables/yellow_taxi2.csv")

#### Analyzing and Understanding Data set

In [0]:
df_csv2.display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
1,7/1/2021 0:30,7/1/2021 0:35,N,1,74,168,1,1.2,6.0,0.5,0.5,0.0,0.0,null,0.3,7.3,2,1,0.0
2,7/1/2021 0:25,7/1/2021 1:01,N,1,116,265,2,13.69,42.0,0.5,0.5,0.0,0.0,null,0.3,43.3,2,1,0.0
2,7/1/2021 0:05,7/1/2021 0:12,N,1,97,33,1,0.95,6.5,0.5,0.5,2.34,0.0,null,0.3,10.14,1,1,0.0
2,7/1/2021 0:41,7/1/2021 0:47,N,1,74,42,1,1.24,6.5,0.5,0.5,0.0,0.0,null,0.3,7.8,2,1,0.0
2,7/1/2021 0:51,7/1/2021 0:58,N,1,42,244,1,1.1,7.0,0.5,0.5,0.0,0.0,null,0.3,8.3,2,1,0.0
1,7/1/2021 0:05,7/1/2021 0:11,N,1,24,239,1,1.9,8.0,3.25,0.5,3.0,0.0,null,0.3,15.05,1,1,2.75
2,7/1/2021 0:57,7/1/2021 1:27,N,1,75,243,1,0.0,17.5,0.5,0.5,0.0,0.0,null,0.3,18.8,2,1,0.0
2,7/1/2021 0:27,7/1/2021 0:32,N,1,82,82,1,0.66,5.0,0.5,0.5,0.0,0.0,null,0.3,6.3,2,1,0.0
2,7/1/2021 0:29,7/1/2021 0:34,N,1,74,42,1,1.72,7.0,0.5,0.5,2.08,0.0,null,0.3,10.38,1,1,0.0
2,7/1/2021 0:41,7/1/2021 0:49,N,1,41,42,1,1.37,7.5,0.5,0.5,0.0,0.0,null,0.3,8.8,2,1,0.0


In [0]:
df_csv2.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: string (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



####1. Calculate the average trip distance for each day of the week.

In [0]:
from pyspark.sql.functions import col, date_format, dayofweek, avg,to_date

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, dayofweek, avg, date_format,format_number





# Convert the lpep_pickup_datetime column to a timestamp type
df_csv2 = df_csv2.withColumn("lpep_timestamp", to_timestamp("lpep_pickup_datetime", "M/d/yyyy H:m"))

# Extract the day of the week from the timestamp column
day_of_week = df_csv2.withColumn("day_of_week", date_format("lpep_timestamp", "EEEE"))

# Calculate the average trip distance per day of the week
result = day_of_week.groupBy("day_of_week") \
                    .agg(format_number(avg("trip_distance"),2).alias("avg_trip_distance"))

# Show the result
result.show()
#df_csv2.show()


+-----------+-----------------+
|day_of_week|avg_trip_distance|
+-----------+-----------------+
|  Wednesday|           197.90|
|    Tuesday|           196.68|
|     Friday|           146.79|
|   Thursday|           186.32|
|   Saturday|           206.43|
|     Monday|           238.26|
|     Sunday|           213.70|
+-----------+-----------------+



####2.	Find the total number of trips for each combination of vendor and payment type.

In [0]:
# Filter out rows where VendorID and payment_type are not null
df1 = df_csv2.where(col("VendorID").isNotNull() & col("payment_type").isNotNull())



# Group by vendor and payment type, and count the number of trips
result = df1.groupBy("VendorID", "payment_type") \
                .count() \
                .withColumnRenamed("count", "total_trips") \
                .orderBy(col("total_trips").desc()) # Order the result by total_trips in descending order

result.show()

+--------+------------+-----------+
|VendorID|payment_type|total_trips|
+--------+------------+-----------+
|       2|           1|      25516|
|       2|           2|      17897|
|       1|           1|       4474|
|       1|           2|       2934|
|       1|           3|        185|
|       2|           3|        122|
|       1|           4|         25|
|       2|           4|         19|
|       1|           5|          1|
+--------+------------+-----------+



###3.	Identify the top 5 pickup locations based on the total fare amount collected. (Make use of the column PULocationID).

In [0]:
from pyspark.sql.functions import col, sum,format_number



# Group by PULocationID, calculate the sum of fare_amount for each location,
# and alias the sum as "total_fare_amount"
result = df_csv2.groupBy("PULocationID") \
                .agg(sum("fare_amount").alias("total_fare_amount")) \
                .orderBy(col("total_fare_amount").desc()) \
                .select("PULocationID", format_number("total_fare_amount", 2).alias("total_fare_amount")) \
                .limit(5)
result.show()

+------------+-----------------+
|PULocationID|total_fare_amount|
+------------+-----------------+
|          74|       105,957.74|
|          75|        97,567.41|
|          41|        59,257.50|
|          42|        46,878.76|
|         244|        44,870.12|
+------------+-----------------+



###4.	Calculate the average tip amount per passenger count for trips where the fare amount is greater than $20.

In [0]:
from pyspark.sql.functions import avg, format_number
df5 = df_csv2.filter((col("passenger_count") > 0)  & (col("fare_amount") > 20))
# Group by passenger_count and calculate average tip_amount
result = df5.groupBy("passenger_count") \
                .agg(avg("tip_amount").alias("avg_tip_amount"))

# Format the average tip_amount to 2 decimal places
result = result.withColumn("avg_tip_amount", format_number("avg_tip_amount", 2)).orderBy(col("passenger_count").asc())

# Select the necessary columns
result = result.select("passenger_count", "avg_tip_amount")

# Show the result
result.show()


+---------------+--------------+
|passenger_count|avg_tip_amount|
+---------------+--------------+
|              1|          2.26|
|              2|          1.84|
|              3|          2.97|
|              4|          3.24|
|              5|          3.05|
|              6|          2.83|
|              7|          5.00|
|             32|          0.00|
+---------------+--------------+



###5.	Find the percentage of trips where the total amount is within 10% of the fare amount.

In [0]:
from pyspark.sql.functions import col, format_number
filtered_df = df_csv2.filter(((col("total_amount") - col("fare_amount")) / col("fare_amount")) * 100 < 10)
# Count the number of rows that satisfy the condition
result = filtered_df.count()
result # total number of records having total_amount is within 10% of fare_amount
original_count = df_csv2.count()
original_count # total number of records in our original dataset
final_result = round((result / original_count) * 100 ,2)
final_result

Out[11]: 23.0